# Session 3 – Preparation: Pandas I

## Setup

pandas builds on top of **NumPy**, so we need both libraries. If you work with Anaconda, they are usually preinstalled – otherwise install them **once** in your terminal:

```bash
pip install numpy pandas
```

In a notebook you can do the same with the `%pip` magic – run the cell below and check that everything works:

In [ ]:
%pip install numpy pandas

import numpy as np
import pandas as pd

print('NumPy ', np.__version__)
print('pandas', pd.__version__)

## Why pandas?

Remember Session 2? You analyzed your café survey with a **list of dictionaries** – and needed loops and helper functions for everything:

```python
for person in survey:
    total += person['visits']
```

That works for 7 guests – but imagine **10,000 orders**. Counting, filtering and averaging by hand becomes painful and slow.

**NumPy** brings fast number processing to Python, and **pandas** builds on top of it: a DataFrame gives your data a table structure. Almost everything you did with loops last session becomes **one line**:

```python
orders['cups_sold'].sum()
```

Let's take a look:

In [ ]:
import pandas as pd

survey = [
    {'name': 'Alice',   'age': 21, 'item': 'coffee', 'visits': 6},
    {'name': 'Bob',     'age': 23, 'item': 'tea',    'visits': 4},
    {'name': 'Charlie', 'age': 22, 'item': 'coffee', 'visits': 8},
]

guests = pd.DataFrame(survey)
print(guests)                                   # a real table
print()
print('Total visits:', guests['visits'].sum()) # no loop needed!
print('Average age: ', guests['age'].mean())

## 1. Getting Started

- `import pandas as pd` – used to import pandas into the notebook
- pandas has two core objects: **Series** (one column) and **DataFrame** (a table)
- Every DataFrame has an **index** – the row labels on the left. By default: 0, 1, 2, ...

In [ ]:
import pandas as pd      # usually at the top of every notebook

cups = pd.Series([120, 98, 130], name='cups_sold')
print(cups)          # a Series: one column with an index
print()
print(cups.sum())   # 348

## 2. DataFrames & Series

The most common ways to meet a DataFrame:

- From a **list of dictionaries** (you know this from Session 2): every dict becomes a row
- From a **dictionary of lists**: every key becomes a column
- From a **CSV file** – the classic case, that's next

In [ ]:
# From a dictionary of lists: every key becomes a column
city_guests = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'visits': [6, 4, 8],
})
print(city_guests)

# One column of a DataFrame is a Series
print(type(city_guests['visits']))
print(type(city_guests))

## 3. Read CSV

CSV (comma-separated values) is **the** standard format for tabular data. First we create our practice dataset: the sales log of one busy week. Then we read it with `pd.read_csv()`.

⚠️ Note: one entry is missing (we lost Wednesday's coffee count). Keep that in mind for the next section.

In [ ]:
csv_text = '''day,item,price,cups_sold
Mon,coffee,2.50,120
Mon,tea,2.20,45
Tue,coffee,2.50,98
Tue,cake,3.90,17
Wed,coffee,2.50,
Wed,tea,2.20,40
Thu,coffee,2.50,130
Thu,cake,3.90,15
Fri,tea,2.20,52
Fri,coffee,2.50,110
Sat,cake,3.90,32
Sat,coffee,2.50,145
'''

with open('cafe_sales.csv', 'w') as f:
    f.write(csv_text)

print('cafe_sales.csv created')

In [ ]:
sales = pd.read_csv('cafe_sales.csv')
print(sales.head())      # first 5 rows – the standard first look
print()
print(sales.tail(3))     # the last rows

## 4. Analyzing Data (and spotting missing values)

Before any analysis: get to know your data!

- `info()` – columns, data types, non-null counts
- `columns` / `index` / `dtypes` – the column names, the row labels and the data types
- `shape` – (number of rows, number of columns)
- `isnull().sum()` – counts the missing values per column

Missing values show up as `NaN` ("not a number"). **Spotting** them is your job today – **handling** them (delete or fill?) is exactly the topic of Session 4.

In [ ]:
sales.info()                  # columns, dtypes, non-null counts
print()
print(sales.columns)           # the column names
print(sales.index)             # the row labels (the index)
print(sales.dtypes)            # the data type of every column
print(sales.shape)             # (12 rows, 4 columns)
print()
print(sales.isnull().sum())    # missing values per column

## 5. Selecting Data

- One column: `sales['item']` → a **Series**
- Several columns: `sales[['day', 'item']]` → a **DataFrame** (note the double brackets!)
- Rows by position: `.iloc[0]` or ranges like `.iloc[0:3]`
- Rows and columns by position: `.iloc[0:3, 0:2]`
- By label: `.loc[0:2, 'day':'price']`

In [ ]:
# One column -> Series
print(sales['item'])

# Several columns -> DataFrame (double brackets!)
print(sales[['day', 'item']])

# Rows by position
print(sales.iloc[0])       # first row
print(sales.iloc[0:3])     # rows 0, 1, 2 – stop excluded, like lists
print(sales.iloc[0:3, 0:2])          # rows AND columns by position
print(sales.loc[0:2, 'day':'price'])  # by label – stop INCLUDED

## 6. Filtering Data

Boolean conditions pick the rows you want:

- `sales[sales['item'] == 'coffee']`
- `sales[sales['cups_sold'] > 100]`
- Combined with `&` (and) / `|` (or) / `~` (not) – **each condition needs its own brackets**
- Several values at once: `.isin([...])`

In [ ]:
# All coffee rows
print(sales[sales['item'] == 'coffee'])

# Busy entries: more than 100 cups
print(sales[sales['cups_sold'] > 100])

# Combined: coffee AND more than 100 cups – brackets around each condition!
print(sales[(sales['item'] == 'coffee') & (sales['cups_sold'] > 100)])

# Several values at once
print(sales[sales['day'].isin(['Sat', 'Sun'])])

# Negation: everything EXCEPT coffee
print(sales[~(sales['item'] == 'coffee')])

## 7. Summary Statistics

- `describe()` – the all-rounder: count, mean, std, min, quartiles, max
- Single measures: `.mean()`, `.sum()`, `.min()`, `.max()`, `.median()`
- `nunique()` / `unique()` – how many distinct values there are – and which ones
- `value_counts()` – how often each value occurs. Perfect for categories like `item`

Summary statistics answer your café questions in one line each.

In [ ]:
print(sales['cups_sold'].describe())

print()
print('Total cups:', sales['cups_sold'].sum())
print('Best entry:', sales['cups_sold'].max())
print()
print(sales['item'].nunique(), 'different items')   # 3
print(sales['item'].unique())                        # which ones
print()
print(sales['item'].value_counts())    # how often each item appears

# The full row of the best-selling entry – filtering + loc combined
print(sales.loc[sales['cups_sold'].idxmax()])